In [9]:
import pandas as pd
import requests
import time
import datetime
import os
import csv

try:
    from google.colab import userdata
    my_api_key = userdata.get('RapidAPI')
except ImportError:
    from dotenv import load_dotenv
    load_dotenv(override=True)
    my_api_key = os.getenv("RAPIDAPI_KEY")

def scraping(query_list, output_csv, target_count=5, is_official_account=False):

    seen_video_ids = set()

    if is_official_account:
        print(f"\nSearch by account : {len(query_list)} accounts")
       
        url = "https://tiktok-api6.p.rapidapi.com/user/videos"
        headers = {
            "x-rapidapi-host": "tiktok-api6.p.rapidapi.com",
            "x-rapidapi-key": my_api_key
        }
        csv_columns = ["target_account", "video_id", "real_video_url", "create_time", 
                       "publish_date", "video_text", "author_name", "play_count", "hearts_count"]
    else:
        print(f"\nSearch by keyword: {len(query_list)} keywords")
        url = "https://tiktok-scraper7.p.rapidapi.com/feed/search"
        headers = {
            "x-rapidapi-host": "tiktok-scraper7.p.rapidapi.com",
            "x-rapidapi-key": my_api_key
        }
        csv_columns = ["search_keyword", "video_id", "real_video_url", "create_time", 
                       "publish_date", "video_text", "author_name", "play_count", "digg_count"]

    if not os.path.exists(output_csv):
        with open(output_csv, 'w', newline='', encoding='utf-8-sig') as f:
            writer = csv.writer(f)
            writer.writerow(csv_columns)

    for item in query_list:
        print(f"Looking for : {item}")
        total_scraped = 0
        current_cursor = "0"
        has_more = True
        retry_count = 0

        

        while has_more and total_scraped < target_count:

            if is_official_account:
                querystring = {"username": item, "cursor": current_cursor}
            else:
                querystring = {"keywords": item, "region": "be", "count": "30", "cursor": current_cursor, "publish_time": "0", "sort_type": "0"}

            try:
                response = requests.get(url, headers=headers, params=querystring)
                
            
                if not is_official_account and response.status_code == 429:
                    retry_count += 1
                    if retry_count > 3: break
                    time.sleep(15)
                    continue
                elif response.status_code != 200:
                    print(f"API Error: {response.status_code}")
                    break

                data = response.json()
                
    
                if is_official_account:
                    videos_list = data.get("data", {}).get("videos", [])
                else:
                    videos_list = data.get("data", {}).get("videos", []) or data.get("data", [])
                
                if not videos_list: break

                for video in videos_list:
                    if not isinstance(video, dict): continue

             
                    raw_id = video.get("video_id") or video.get("aweme_id") or video.get("id")
                    video_id = str(raw_id) if raw_id else ""

                    if video_id and video_id not in seen_video_ids:
                        seen_video_ids.add(video_id)
                        
                        raw_time = video.get("create_time")
                        formatted_date = datetime.datetime.fromtimestamp(int(raw_time)).strftime('%Y-%m-%d %H:%M:%S') if raw_time else ""
                        stats = video.get("statistics", {})

                        with open(output_csv, 'a', newline='', encoding='utf-8-sig') as f:
                            writer = csv.writer(f)
                            if is_official_account:
                               
                                text = str(video.get("title") or video.get("desc") or "").strip()
                                real_url = f"https://www.tiktok.com/@{item}/video/{video_id}"
                                writer.writerow([
                                    item, video_id, real_url, raw_time, formatted_date, text, 
                                    item, stats.get("play_count", 0), stats.get("digg_count", 0)
                                ])
                            else:
                                text = str(video.get("title") or video.get("desc") or "").strip()
                                actual_author = video.get("author", {}).get("unique_id") or video.get("author", {}).get("nickname") or "user"
                                real_url = f"https://www.tiktok.com/@{actual_author}/video/{video_id}"
                                writer.writerow([
                                    item, video_id, real_url, raw_time, formatted_date, text, 
                                    actual_author, stats.get("play_count", 0), stats.get("digg_count", 0)
                                ])
                                
                        total_scraped += 1
                        if total_scraped >= target_count: break

                if is_official_account:
                    data_block = data.get("data", {})
                    next_cursor = data_block.get("cursor")
                    if next_cursor and str(next_cursor) != "0" and str(next_cursor) != current_cursor:
                        current_cursor = str(next_cursor)
                    else:
                        has_more = False
                else:
                    meta_data = data.get("data", {})
                    current_cursor = str(meta_data.get("cursor", int(current_cursor) + 30))
                    has_more = meta_data.get("hasMore", False) or meta_data.get("has_more", False)

            except Exception as e:
                print(f"Error: {e}")
                break
            
            time.sleep(3)



In [11]:
def main():
    try:
        df_no_account = pd.read_excel("/Users/a0979/Desktop/tiktok/no_account/Vlaams Belang_no_account.xlsx")
        keywords_list = df_no_account['search_keyword'].dropna().unique().tolist()
        
        scraping(
            query_list=keywords_list, 
            output_csv="Vlaams_Belang_keywords_search.csv", 
            target_count=1, 
            is_official_account=False
        )
    except FileNotFoundError:
        print("Can't find the file")

    try:
        df_with_account = pd.read_excel("/Users/a0979/Desktop/tiktok/with_account/Vlaams Belang_with_account.xlsx")
        
        usernames_list = []
        for url in df_with_account['Tiktok'].dropna():
            if '@' in str(url):
                username = str(url).split('@')[-1].split('/')[0].split('?')[0]
                usernames_list.append(username)
        usernames_list = list(set(usernames_list))
        
        scraping(
            query_list=usernames_list, 
            output_csv="Vlaams_Belang_official.csv", 
            target_count=1, 
            is_official_account=True
        )
    except FileNotFoundError:
        print("Can't find the file")

if __name__ == "__main__":
    main()


Search by keyword: 11 keywords
Looking for : Immanuel De Reuse Vlaams Belang
Looking for : Marijke Dillen Vlaams Belang
Looking for : Jan Laeremans Vlaams Belang
Looking for : Freya Perdaens Vlaams Belang
Looking for : Wouter Raskin Vlaams Belang
Looking for : Gwendolyn Rutten Vlaams Belang
Looking for : Bart Tommelein Vlaams Belang
Looking for : Servais Verherstraeten Vlaams Belang
Looking for : Annelies Verlinden Vlaams Belang
Looking for : Wouter Vermeersch Vlaams Belang
Looking for : Bert Wollants Vlaams Belang

Search by account : 31 accounts
Looking for : klaasslootmans
API Error: 429
Looking for : ankevandermeersch
API Error: 429
Looking for : dominieksneppe
API Error: 429
Looking for : adelineblancquaer
API Error: 429
Looking for : roosmarijnbeckers8
API Error: 429
Looking for : erik.gilissen
API Error: 429
Looking for : driesvanlangenhove
API Error: 429
Looking for : ellen_samyn
API Error: 429
Looking for : screyelman
API Error: 429
Looking for : reccino
API Error: 429
Lookin